# Customer Segmentation and Sales Analytics using K-Means Clustering

**IBM SkillsBuild Data Analytics with AI Internship**

---

## Project Overview

This notebook performs end-to-end customer segmentation on a retail sales dataset.  
The workflow covers data loading, cleaning, exploratory data analysis (EDA), feature engineering,  
K-Means clustering, visualisation, and business insight generation.

**Sections:**
1. Import Libraries
2. Load Dataset
3. Data Quality Check
4. Data Cleaning
5. Exploratory Data Analysis (EDA)
6. Customer Purchasing Behaviour Analysis (RFM)
7. Feature Preparation
8. K-Means Clustering & Elbow Method
9. Cluster Visualisation
10. Cluster Interpretation
11. Business Insights & Recommendations
12. Conclusion

> **Before running:** Place your dataset CSV inside the `data/` folder and update `DATA_FILE` in Section 2.

---
## Section 1 — Import Libraries

In [1]:
# Standard library
import warnings
import os

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Settings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Ensure output directory exists
os.makedirs('outputs', exist_ok=True)

print('All libraries imported successfully.')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'sklearn : ', end='')
import sklearn; print(sklearn.__version__)

ImportError: DLL load failed while importing _openmp_helpers: An Application Control policy has blocked this file.

---
## Section 2 — Load Dataset

> **Action required:** Update `DATA_FILE` below with your actual filename (e.g. `'data/online_retail.csv'`).

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
DATA_FILE = 'data/online_retail.csv'-  # <-- UPDATE THIS
# ───────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_FILE, encoding='unicode_escape')

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumn names:')
print(df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

---
## Section 3 — Data Quality Check

Before cleaning, we inspect the dataset for:
- Shape and data types
- Missing values
- Duplicate rows
- Basic descriptive statistics

In [ ]:
print('=== Shape ===')
print(f'{df.shape[0]:,} rows, {df.shape[1]} columns')

print('\n=== Data Types ===')
print(df.dtypes)

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df if not missing_df.empty else 'No missing values found.')

In [ ]:
print('=== Duplicate Rows ===')
dup_count = df.duplicated().sum()
print(f'{dup_count:,} duplicate rows found ({dup_count / len(df) * 100:.2f}%)')

print('\n=== Descriptive Statistics ===')
df.describe()

---
## Section 4 — Data Cleaning

Based on the quality check above, we will:
1. Drop duplicate rows
2. Handle missing values (drop or impute as appropriate)
3. Fix data types (parse dates, ensure numeric types)
4. Remove records with invalid quantities or prices (negative / zero values)
5. Create a `TotalPrice` column (`Quantity × UnitPrice`)

> **Note:** Column name mapping below. Update the dictionary if your dataset uses different names.

In [ ]:
# ── Column name mapping ─────────────────────────────────────────────────────────
# Update these keys to match your actual column names
COL_CUSTOMER_ID  = 'CustomerID'    # Unique customer identifier
COL_INVOICE_DATE = 'InvoiceDate'   # Transaction date
COL_QUANTITY     = 'Quantity'      # Units purchased
COL_UNIT_PRICE   = 'UnitPrice'     # Price per unit
COL_INVOICE_NO   = 'InvoiceNo'     # Invoice / order identifier
COL_DESCRIPTION  = 'Description'   # Product description
# ───────────────────────────────────────────────────────────────────────────────

df_clean = df.copy()

# Step 1 — Drop duplicates
before = len(df_clean)
df_clean.drop_duplicates(inplace=True)
print(f'Duplicates removed : {before - len(df_clean):,} rows')

# Step 2 — Drop rows with missing CustomerID (cannot segment without it)
before = len(df_clean)
df_clean.dropna(subset=[COL_CUSTOMER_ID], inplace=True)
print(f'Rows without CustomerID removed : {before - len(df_clean):,} rows')

# Step 3 — Parse invoice date
df_clean[COL_INVOICE_DATE] = pd.to_datetime(df_clean[COL_INVOICE_DATE], errors='coerce')
df_clean.dropna(subset=[COL_INVOICE_DATE], inplace=True)
print(f'Date column parsed. Remaining rows : {len(df_clean):,}')

In [ ]:
# Step 4 — Remove returns (negative quantity) and invalid prices
before = len(df_clean)
df_clean = df_clean[(df_clean[COL_QUANTITY] > 0) & (df_clean[COL_UNIT_PRICE] > 0)]
print(f'Invalid quantity/price rows removed : {before - len(df_clean):,} rows')

# Step 5 — Create TotalPrice column
df_clean['TotalPrice'] = df_clean[COL_QUANTITY] * df_clean[COL_UNIT_PRICE]

print(f'\nClean dataset shape : {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns')
df_clean.head()

---
## Section 5 — Exploratory Data Analysis (EDA)

We explore the cleaned data to understand:
- Distribution of order quantities and transaction values
- Sales trend over time
- Top products by revenue
- Correlation between numeric features

In [ ]:
# --- Distribution of Quantity and TotalPrice ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cap at 95th percentile for readability
q95_qty   = df_clean[COL_QUANTITY].quantile(0.95)
q95_price = df_clean['TotalPrice'].quantile(0.95)

sns.histplot(df_clean[df_clean[COL_QUANTITY] <= q95_qty][COL_QUANTITY],
             bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Order Quantity (≤ 95th pct)')
axes[0].set_xlabel('Quantity')

sns.histplot(df_clean[df_clean['TotalPrice'] <= q95_price]['TotalPrice'],
             bins=40, ax=axes[1], color='coral')
axes[1].set_title('Distribution of Total Price (≤ 95th pct)')
axes[1].set_xlabel('Total Price')

plt.tight_layout()
plt.savefig('outputs/eda_distributions.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eda_distributions.png')

In [ ]:
# --- Monthly Sales Trend ---
df_clean['YearMonth'] = df_clean[COL_INVOICE_DATE].dt.to_period('M')
monthly_sales = df_clean.groupby('YearMonth')['TotalPrice'].sum().reset_index()
monthly_sales['YearMonth'] = monthly_sales['YearMonth'].astype(str)

plt.figure(figsize=(12, 4))
sns.lineplot(data=monthly_sales, x='YearMonth', y='TotalPrice', marker='o', color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Sales Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('outputs/eda_monthly_sales.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eda_monthly_sales.png')

In [ ]:
# --- Top 10 Products by Revenue ---
top_products = (
    df_clean.groupby(COL_DESCRIPTION)['TotalPrice']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=top_products, x='TotalPrice', y=COL_DESCRIPTION, palette='Blues_r')
plt.title('Top 10 Products by Revenue')
plt.xlabel('Total Revenue')
plt.ylabel('Product')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('outputs/eda_top_products.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eda_top_products.png')

In [ ]:
# --- Correlation Heatmap ---
numeric_cols = df_clean.select_dtypes(include='number')

plt.figure(figsize=(7, 5))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('outputs/eda_correlation.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eda_correlation.png')

---
## Section 6 — Customer Purchasing Behaviour Analysis (RFM)

We build **RFM features** — a widely used framework in customer analytics:

| Feature | Definition |
|---|---|
| **Recency (R)** | Days since the customer's last purchase (lower = more recent) |
| **Frequency (F)** | Total number of invoices / transactions made |
| **Monetary (M)** | Total amount spent across all transactions |

In [ ]:
# Reference date: one day after the last transaction in the dataset
reference_date = df_clean[COL_INVOICE_DATE].max() + pd.Timedelta(days=1)
print(f'Reference date for Recency calculation: {reference_date.date()}')

rfm = df_clean.groupby(COL_CUSTOMER_ID).agg(
    Recency   = (COL_INVOICE_DATE, lambda x: (reference_date - x.max()).days),
    Frequency = (COL_INVOICE_NO,   'nunique'),
    Monetary  = ('TotalPrice',     'sum')
).reset_index()

rfm.columns = [COL_CUSTOMER_ID, 'Recency', 'Frequency', 'Monetary']
print(f'\nRFM table shape: {rfm.shape}')
rfm.head(10)

In [ ]:
# --- RFM Distributions ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rfm_cols   = ['Recency', 'Frequency', 'Monetary']
colors     = ['#4C72B0', '#55A868', '#C44E52']

for ax, col, color in zip(axes, rfm_cols, colors):
    # Cap Monetary at 95th pct for readability
    plot_data = rfm[rfm[col] <= rfm[col].quantile(0.95)][col]
    sns.histplot(plot_data, bins=30, ax=ax, color=color)
    ax.set_title(f'{col} Distribution (≤ 95th pct)')
    ax.set_xlabel(col)

plt.tight_layout()
plt.savefig('outputs/eda_rfm_distributions.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/eda_rfm_distributions.png')

In [ ]:
# --- Summary statistics for RFM ---
print('RFM Summary Statistics:')
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)

---
## Section 7 — Feature Preparation

K-Means is a distance-based algorithm; features on different scales will bias the clustering.  
We apply **StandardScaler** (zero mean, unit variance) to the RFM features before fitting.

In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
X = rfm[features].copy()

scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print('Feature scaling complete (StandardScaler).')
print('\nScaled feature summary:')
X_scaled_df.describe().round(3)

---
## Section 8 — K-Means Clustering & Elbow Method

### 8.1 Elbow Method
We fit K-Means for **k = 1 to 10** and plot the **inertia** (within-cluster sum of squares).  
The optimal k is the point where adding another cluster gives diminishing returns — the "elbow".

### 8.2 Silhouette Score
We also compute the Silhouette Score for each k to confirm the choice quantitatively.

In [ ]:
K_RANGE = range(1, 11)
inertias        = []
silhouette_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    if k > 1:  # Silhouette requires at least 2 clusters
        score = silhouette_score(X_scaled, km.labels_)
        silhouette_scores.append(score)
    else:
        silhouette_scores.append(None)

print('Inertia values:')
for k, inertia in zip(K_RANGE, inertias):
    print(f'  k={k:2d}  inertia={inertia:,.2f}')

In [ ]:
# --- Elbow Curve ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Inertia
axes[0].plot(list(K_RANGE), inertias, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('Elbow Method — Inertia vs. k')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_xticks(list(K_RANGE))

# Silhouette Score
sil_k      = list(K_RANGE)[1:]          # starts at k=2
sil_scores = [s for s in silhouette_scores if s is not None]
axes[1].plot(sil_k, sil_scores, marker='s', color='coral', linewidth=2)
axes[1].set_title('Silhouette Score vs. k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(sil_k)

plt.tight_layout()
plt.savefig('outputs/elbow_curve.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/elbow_curve.png')

In [ ]:
# ── Set optimal k after reviewing the plots above ──────────────────────────────
OPTIMAL_K = 4    # <-- UPDATE based on the elbow / silhouette charts
# ───────────────────────────────────────────────────────────────────────────────

kmeans_final = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=10, random_state=42)
kmeans_final.fit(X_scaled)

rfm['Cluster'] = kmeans_final.labels_

print(f'K-Means fitted with k = {OPTIMAL_K}')
print(f'Final inertia : {kmeans_final.inertia_:,.2f}')
print(f'Silhouette Score : {silhouette_score(X_scaled, kmeans_final.labels_):.4f}')
print('\nCluster counts:')
print(rfm['Cluster'].value_counts().sort_index())

---
## Section 9 — Cluster Visualisation

Since the RFM feature space is 3-dimensional, we use **PCA** to reduce it to 2 dimensions  
solely for visualisation purposes. The clustering itself was performed in the full 3-D space.

In [ ]:
# --- PCA Reduction for visualisation only ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = rfm['Cluster'].values

print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.2%},',
      f'PC2={pca.explained_variance_ratio_[1]:.2%}')
print(f'Total variance explained by 2 components: {sum(pca.explained_variance_ratio_):.2%}')

In [ ]:
# --- Scatter Plot: Clusters in PCA space ---
palette = sns.color_palette('tab10', n_colors=OPTIMAL_K)

plt.figure(figsize=(9, 6))
for cluster_id in sorted(pca_df['Cluster'].unique()):
    subset = pca_df[pca_df['Cluster'] == cluster_id]
    plt.scatter(subset['PC1'], subset['PC2'],
                label=f'Cluster {cluster_id}',
                alpha=0.6, s=20, color=palette[cluster_id])

plt.title(f'K-Means Cluster Visualisation (PCA 2-D) — k={OPTIMAL_K}')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend(title='Cluster', loc='best')
plt.tight_layout()
plt.savefig('outputs/cluster_scatter.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/cluster_scatter.png')

In [ ]:
# --- Box plots: RFM distributions per cluster ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, color in zip(axes, ['Recency', 'Frequency', 'Monetary'], ['#4C72B0', '#55A868', '#C44E52']):
    rfm_plot = rfm[rfm[col] <= rfm[col].quantile(0.99)]  # remove extreme outliers for display
    sns.boxplot(data=rfm_plot, x='Cluster', y=col, ax=ax, palette='tab10')
    ax.set_title(f'{col} by Cluster')
    ax.set_xlabel('Cluster')

plt.suptitle('RFM Feature Distributions per Cluster', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('outputs/cluster_rfm_boxplots.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/cluster_rfm_boxplots.png')

---
## Section 10 — Cluster Interpretation

We compute per-cluster summary statistics to understand what each segment represents.  
After running this section, assign meaningful labels to each cluster in the cell below.

In [ ]:
# --- Per-cluster mean RFM statistics ---
cluster_profiles = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].agg(
    ['mean', 'median', 'std', 'count']
).round(2)

cluster_profiles.columns = ['_'.join(col) for col in cluster_profiles.columns]
cluster_profiles = cluster_profiles.reset_index()

print('Cluster Profiles (mean / median / std / count):')
cluster_profiles

In [ ]:
# --- Save cluster profiles ---
cluster_profiles.to_csv('outputs/cluster_profiles.csv', index=False)
print('Saved → outputs/cluster_profiles.csv')

In [ ]:
# ── Assign segment labels after reviewing cluster profiles above ────────────────
# Update these labels to reflect what you observe in your data.
# Example labels — replace with your actual interpretation:
CLUSTER_LABELS = {
    0: 'Segment A — [Your Label Here]',
    1: 'Segment B — [Your Label Here]',
    2: 'Segment C — [Your Label Here]',
    3: 'Segment D — [Your Label Here]',
}
# ───────────────────────────────────────────────────────────────────────────────

rfm['Segment'] = rfm['Cluster'].map(CLUSTER_LABELS)

print('Segment distribution:')
print(rfm['Segment'].value_counts())

---
## Section 11 — Business Insights & Recommendations

> **Instructions:** After running the notebook and reviewing the cluster profiles, replace the
> placeholder text below with your actual findings. Each sub-section corresponds to one cluster.

---

### 11.1 Segment-wise Analysis

#### Cluster 0 — [Label]
- **Profile:** *(describe recency, frequency, monetary characteristics)*
- **Who they are:** *(e.g., loyal high-value customers, infrequent buyers, etc.)*
- **Recommended strategy:** *(e.g., VIP loyalty programme, personalised offers, win-back emails)*

#### Cluster 1 — [Label]
- **Profile:** *(describe characteristics)*
- **Who they are:** *(description)*
- **Recommended strategy:** *(action)*

#### Cluster 2 — [Label]
- **Profile:** *(describe characteristics)*
- **Who they are:** *(description)*
- **Recommended strategy:** *(action)*

#### Cluster 3 — [Label]
- **Profile:** *(describe characteristics)*
- **Who they are:** *(description)*
- **Recommended strategy:** *(action)*

---

### 11.2 Overall Strategic Recommendations

1. **Prioritise high-value segments** — Focus retention resources on the most profitable segment.
2. **Re-engage at-risk customers** — Time-limited campaigns targeting low-recency, mid-spend groups.
3. **Nurture new customers** — Onboarding discounts and product recommendations for recent first-time buyers.
4. **Track segment migration** — Re-run clustering periodically to monitor customers shifting between segments.
5. **Personalise communications** — Use segment labels in CRM tools for targeted email and ad campaigns.

---
## Section 12 — Conclusion

> **Instructions:** Fill in this section after completing the analysis.

---

### Summary of Findings

In this project, we applied K-Means Clustering to a retail sales dataset to segment customers
based on their purchasing behaviour (Recency, Frequency, Monetary — RFM metrics).

- **Dataset:** *(describe — rows, columns, time period)*
- **Cleaning:** *(describe what was removed/fixed)*
- **Optimal k:** *(k = ?, justified by Elbow Method and Silhouette Score)*
- **Segments identified:** *(list and briefly describe each)*

### Objectives Achieved

| Objective | Status |
|---|---|
| Data loading and quality check | ✅ Completed |
| Data cleaning | ✅ Completed |
| Exploratory Data Analysis | ✅ Completed |
| RFM feature engineering | ✅ Completed |
| K-Means clustering with Elbow Method | ✅ Completed |
| Cluster visualisation | ✅ Completed |
| Business insights and recommendations | ✅ Completed |

### Future Work

- Explore **DBSCAN** or **Hierarchical Clustering** for non-spherical cluster shapes.
- Incorporate demographic data to enrich segment profiles.
- Build a **churn prediction model** on top of the segments.
- Automate the pipeline for real-time customer segmentation updates.

---

*End of Notebook — IBM SkillsBuild Data Analytics with AI Internship*